# Part 1 · Anatomy of a molecular crystal  (~8 min)

Paracetamol has two well-known crystal forms:

| | Form I | Form II |
|---|---|---|
| Space group | P2₁/n (monoclinic) | Pbca (orthorhombic) |
| Stability at room T | stable (the tablet form) | metastable |
| Tabletting | poor (brittle) | good (has slip planes) |

Structures from the Crystallography Open Database (COD 7126913 and 7105574, 100 K).

In [ ]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit "requests==2.32.4"
import csp_utils as cu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

## 1.1 Look at the two crystals
Drag to rotate, scroll to zoom. The brown box is one unit cell.

In [ ]:
form1 = cu.load_cif(cu.DATA / "paracetamol_form_I.cif")
form2 = cu.load_cif(cu.DATA / "paracetamol_form_II.cif")
print(form1.get_chemical_formula(), "|", form2.get_chemical_formula())
cu.show(form1, supercell=(2, 2, 1))

In [ ]:
cu.show(form2, supercell=(1, 2, 1))

## 1.2 The numbers that describe a packing

* Z: molecules per unit cell; Z′: molecules in the asymmetric unit (1 for both forms)
* density and packing coefficient (fraction of space filled; organic crystals are usually 0.65–0.77)
* hydrogen bonds: two donors (O–H, N–H) and two acceptors (O=C, O–H)

In [ ]:
table = pd.DataFrame([cu.describe(form1, name="Form I"), cu.describe(form2, name="Form II")]).set_index("name").T
table

In [ ]:
pd.DataFrame(cu.hydrogen_bonds(form1)).query("intermolecular")

> **Question 1.** Which form is denser? Form I is the stable one. Does paracetamol follow the density rule?

## 1.3 Which form is lower in energy?

Lattice energy per molecule:

$$E_\text{latt} = \frac{E_\text{crystal}}{Z} - E_\text{gas molecule}$$

Energies from MACE-OFF, a machine-learned force field for organic molecules (Kovács *et al.*, *JACS* 2025).
Both structures were relaxed with it beforehand.

In [ ]:
from ase.io import read
relaxed = {n: read(cu.DATA / f"paracetamol_form_{n}_relaxed.extxyz") for n in ("I", "II")}
e_gas = cu.gas_phase_energy()
rows = []
for n, at in relaxed.items():
    e = cu.energy_per_molecule(at)
    rows.append({"form": n, "E_latt (kJ/mol)": round(e - e_gas, 1), "density relaxed": round(cu.density(at), 3),
                 "density exp (100 K)": round(cu.density(form1 if n == "I" else form2), 3)})
pd.DataFrame(rows)

> **Question 2.** Experimentally Form I is more stable by only 1–2 kJ/mol. Does MACE-OFF get the order right?
> Compare the difference with one hydrogen bond (~20 kJ/mol).